In [ ]:
# ============================================================
# SHAP EXPLANATION (all 4 classes, per sample image)
# ============================================================
!pip install shap --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ---- predict function (black-box, avoids nested-model gradient tracing issues) ----
def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

# ---- pick 1 (or more) sample per class so all 4 classes are covered ----
SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

sample_imgs = []
sample_true_labels = []
for idx in selected_indices:
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    sample_imgs.append(img.numpy())
    sample_true_labels.append(true_label)
sample_imgs = np.array(sample_imgs)

# ---- masker + explainer (perturbation-based, works with any black-box model) ----
masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
shap_explainer = shap.Explainer(predict_fn, masker, output_names=class_names)

SHAP_MAX_EVALS = 300  # dorkar hole barao (accuracy up but time up)

shap_values = shap_explainer(
    sample_imgs,
    max_evals=SHAP_MAX_EVALS,
    batch_size=50,
    outputs=shap.Explanation.argsort.flip[:len(class_names)]
)

# ---- plot: every row = 1 image, columns = attribution map per class ----
shap.image_plot(shap_values, sample_imgs / 255.0, show=False)
plt.savefig("/kaggle/working/shap_all_classes.png", dpi=600, bbox_inches="tight")
plt.show()

# ---- confidence per class ----
preds_all = predict_fn(sample_imgs)
shap_confidence_records = []
for i, idx in enumerate(selected_indices):
    print(f"Image {i} (index {idx}, true: {class_names[sample_true_labels[i]]}):")
    for c, cname in enumerate(class_names):
        conf = float(preds_all[i, c])
        print(f"   {cname}: {conf*100:.2f}%")
        shap_confidence_records.append({
            "image_index": idx,
            "true_label": class_names[sample_true_labels[i]],
            "explained_class": cname,
            "confidence": conf
        })

pd.DataFrame(shap_confidence_records).to_csv("/kaggle/working/shap_confidence.csv", index=False)
print("Saved: shap_all_classes.png, shap_confidence.csv")


In [ ]:
# ============================================================
# SHAP EXPLANATION (all 4 classes, per sample image) -- FIXED
#
# Fixes vs. original Cell 5:
#   1. shap.image_plot() uses ONE global color scale across ALL
#      rows/classes -> when the model is very confident, non-
#      predicted classes get near-zero SHAP values and become
#      invisible (this is exactly what you saw). Fixed by
#      plotting manually with PER-ROW (per-image) normalization,
#      so each image's own class comparison is visible.
#   2. Caption now clearly separates "Model confidence" (classifier
#      output) from "Mean |SHAP|" (SHAP's own attribution magnitude
#      for that class) -- these are two different numbers.
#   3. CSV records both columns explicitly.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import shap

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

sample_imgs = []
sample_true_labels = []
for idx in selected_indices:
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    sample_imgs.append(img.numpy())
    sample_true_labels.append(true_label)
sample_imgs = np.array(sample_imgs)

masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
shap_explainer = shap.Explainer(predict_fn, masker, output_names=class_names)

SHAP_MAX_EVALS = 300

shap_values = shap_explainer(
    sample_imgs,
    max_evals=SHAP_MAX_EVALS,
    batch_size=50,
    outputs=shap.Explanation.argsort.flip[:len(class_names)]
)
# shap_values.values shape: (n_images, H, W, C, n_classes)
# shap_values.output_names gives the class name per output slot, PER IMAGE
# (because outputs were sorted most-likely-first per image)

preds_all = predict_fn(sample_imgs)

n_rows = len(selected_indices)
n_cols = len(class_names) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

shap_records = []

for row, idx in enumerate(selected_indices):
    img_disp = sample_imgs[row] / 255.0
    axes[row, 0].imshow(img_disp)
    pred_idx = int(np.argmax(preds_all[row]))
    axes[row, 0].set_title(
        f"Original\nTrue: {class_names[sample_true_labels[row]]}\n"
        f"Pred: {class_names[pred_idx]} ({preds_all[row, pred_idx]*100:.2f}%)"
    )
    axes[row, 0].axis("off")

    # Per-row (per-image) heatmaps for ALL classes, channel-summed
    row_heatmaps = {}
    for slot in range(len(class_names)):
        cname = shap_values.output_names[row][slot] if hasattr(shap_values.output_names[row], "__getitem__") \
                else shap_values.output_names[slot]
        heat = shap_values.values[row][..., slot]      # (H, W, C)
        heat = heat.sum(axis=-1)                        # (H, W), signed
        row_heatmaps[cname] = heat

    # PER-IMAGE (per-row) color scale -- this is the key fix.
    row_max_abs = max(np.abs(h).max() for h in row_heatmaps.values()) + 1e-12

    for c, cname in enumerate(class_names):
        heat = row_heatmaps.get(cname, np.zeros(sample_imgs[row].shape[:2]))
        axes[row, c + 1].imshow(img_disp)
        im = axes[row, c + 1].imshow(
            heat, cmap="coolwarm", alpha=0.6,
            vmin=-row_max_abs, vmax=row_max_abs
        )
        mean_abs_shap = float(np.abs(heat).mean())
        model_conf = float(preds_all[row, c])

        axes[row, c + 1].set_title(
            f"{cname}\nModel conf: {model_conf*100:.2f}%\n"
            f"Mean |SHAP|: {mean_abs_shap:.5f}"
        )
        axes[row, c + 1].axis("off")

        shap_records.append({
            "image_index": idx,
            "true_label": class_names[sample_true_labels[row]],
            "pred_label": class_names[pred_idx],
            "explained_class": cname,
            "model_confidence": model_conf,     # classifier's own softmax output
            "mean_abs_shap": mean_abs_shap,      # SHAP's own attribution magnitude
        })

    # small shared colorbar per row
    cbar = fig.colorbar(im, ax=axes[row, 1:].tolist(), shrink=0.7, pad=0.01)
    cbar.set_label("SHAP value (row-normalized)")

plt.savefig("/kaggle/working/shap_all_classes.png", dpi=600, bbox_inches="tight")
plt.show()

pd.DataFrame(shap_records).to_csv("/kaggle/working/shap_confidence.csv", index=False)
print("Saved: shap_all_classes.png, shap_confidence.csv")
print("\nNote: 'model_confidence' = classifier softmax output (NOT SHAP's output).")
print("      'mean_abs_shap'     = SHAP's own attribution magnitude for that class,")
print("      normalized PER IMAGE so non-predicted classes stay visible.")

In [ ]:
# ============================================================
# SHAP - INSERTION / DELETION AUC (patch-based causal metric)
# Session-resumable checkpoint. SHAP perturbation eval o slow,
# tai same small per-class sample set (selected_indices) use kora hocche.
# ============================================================
import os
import cv2
import numpy as np
import pandas as pd
import shap

CHECKPOINT_PATH_SHAP = "/kaggle/working/shap_causal_metrics_progress.csv"
SUMMARY_PATH_SHAP = "/kaggle/working/shap_causal_metrics_summary.csv"

PATCH_SIZE = 16
N_STEPS = 20
SHAP_MAX_EVALS = 300

# ---- utilities (redefined here so this cell runs standalone too) ----
def get_patch_order(heatmap_224, patch_size=16):
    h, w = heatmap_224.shape
    n_h, n_w = h // patch_size, w // patch_size
    patch_scores = np.zeros((n_h, n_w))
    for i in range(n_h):
        for j in range(n_w):
            patch_scores[i, j] = heatmap_224[
                i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size
            ].mean()
    flat_order = np.argsort(-patch_scores.ravel())
    order = np.stack(np.unravel_index(flat_order, patch_scores.shape), axis=1)
    return order

def causal_metric(img_224, heatmap_224, pred_index, mode="insertion",
                   patch_size=16, n_steps=20):
    order = get_patch_order(heatmap_224, patch_size)
    n_patches = len(order)
    step = max(1, n_patches // n_steps)

    blurred = cv2.GaussianBlur(img_224, (31, 31), 0)
    canvas = blurred.copy() if mode == "insertion" else img_224.copy()

    scores, fractions = [], []
    for idx in range(0, n_patches + step, step):
        idx = min(idx, n_patches)
        for (pi, pj) in order[:idx]:
            r0, r1 = pi*patch_size, (pi+1)*patch_size
            c0, c1 = pj*patch_size, (pj+1)*patch_size
            if mode == "insertion":
                canvas[r0:r1, c0:c1] = img_224[r0:r1, c0:c1]
            else:
                canvas[r0:r1, c0:c1] = blurred[r0:r1, c0:c1]
        pred = model.predict(canvas[np.newaxis, ...], verbose=0)[0]
        scores.append(pred[pred_index])
        fractions.append(idx / n_patches)
        if idx == n_patches:
            break
    return float(np.trapz(scores, fractions))

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

# ---- eval set ----
N_EVAL_SAMPLES_SHAP = len(selected_indices)   # ei list ta age visualization cell e create hoyeche
eval_indices = selected_indices[:N_EVAL_SAMPLES_SHAP]

if os.path.exists(CHECKPOINT_PATH_SHAP):
    results_df = pd.read_csv(CHECKPOINT_PATH_SHAP)
    done_indices = set(results_df["image_index"].tolist())
    print(f"Resuming... {len(done_indices)}/{len(eval_indices)} done")
else:
    results_df = pd.DataFrame(columns=[
        "image_index", "true_label", "pred_label", "confidence",
        "SHAP_Insertion_AUC", "SHAP_Deletion_AUC"
    ])
    done_indices = set()

masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
shap_explainer_auc = shap.Explainer(predict_fn, masker, output_names=class_names)

for idx in eval_indices:
    if idx in done_indices:
        continue

    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()

    preds = predict_fn(img_np[np.newaxis, ...])[0]
    pred_idx = int(np.argmax(preds))
    confidence = float(preds[pred_idx])

    # outputs=argsort.flip -> per-sample outputs sorted MOST-LIKELY-FIRST,
    # tai slot 0 = predicted (top) class-er attribution map, always.
    sv = shap_explainer_auc(
        img_np[np.newaxis, ...],
        max_evals=SHAP_MAX_EVALS,
        batch_size=50,
        outputs=shap.Explanation.argsort.flip[:len(class_names)]
    )

    heatmap = sv.values[0][..., 0]          # (H, W, C) - top predicted class
    heatmap = np.abs(heatmap).sum(axis=-1)  # channel-reduce -> (H, W)
    heatmap = heatmap / (heatmap.max() + 1e-8)

    ins_auc = causal_metric(img_np, heatmap, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_auc = causal_metric(img_np, heatmap, pred_idx, "deletion", PATCH_SIZE, N_STEPS)

    results_df.loc[len(results_df)] = [
        idx, class_names[true_label], class_names[pred_idx], confidence, ins_auc, del_auc
    ]
    results_df.to_csv(CHECKPOINT_PATH_SHAP, index=False)
    print(f"[{idx}] true={class_names[true_label]} pred={class_names[pred_idx]} "
          f"conf={confidence*100:.2f}%  Ins={ins_auc:.4f}  Del={del_auc:.4f}")

summary = pd.DataFrame({
    "Mean_Confidence":    [results_df["confidence"].mean()],
    "SHAP_Insertion_AUC": [results_df["SHAP_Insertion_AUC"].mean()],
    "SHAP_Deletion_AUC":  [results_df["SHAP_Deletion_AUC"].mean()],
})
summary.to_csv(SUMMARY_PATH_SHAP, index=False)
print("\n=== SHAP Causal Faithfulness Summary ===")
print(summary.T)
